In [6]:
import pandas as pd

INPUT_CSV = "/kaggle/input/movies/my_data.csv"

try:
    # load data
    df = pd.read_csv(INPUT_CSV)
    
    # print columns
    print("Columns in the dataset:")
    print(df.columns.tolist())
    
except FileNotFoundError:
    print("Kaggle Input not found")

Columns in the dataset:
['budget', 'genres', 'homepage', 'id', 'keywords', 'original_language', 'original_title', 'overview', 'popularity', 'production_companies', 'production_countries', 'release_date', 'revenue', 'runtime', 'spoken_languages', 'status', 'tagline', 'title_x', 'vote_average', 'vote_count', 'cast', 'crew', 'match_title', 'netflix_desc', 'disney_desc', 'amazon_desc', 'hulu_desc', 'streaming_desc', 'title_y', 'poster', 'Series_Title', 'Poster_Link', 'Poster']


In [2]:
import torch
import pandas as pd
import requests
from PIL import Image
from transformers import AutoModelForCausalLM, AutoTokenizer
import os, gc

# ---  ---
INPUT_CSV = "/kaggle/input/movies/my_data.csv"
FINAL_OUTPUT = "movies_full_descriptions.csv"
MODEL_ID = "vikhyatk/moondream2"
NEW_COLUMN = "AI POSTER DESC" # the new desc poster column
SAVE_EVERY = 10 # save every 10 images
device = "cuda" if torch.cuda.is_available() else "cpu"

def clean_vram():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

# 1. loading model
print("--- Loading Moondream Model... ---")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, 
    trust_remote_code=True, 
    torch_dtype=torch.float16
).to(device).eval()
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

# 2. check if file exist (Resume Logic)
if os.path.exists(FINAL_OUTPUT):
    print(f"Loading existing progress from {FINAL_OUTPUT}...")
    df = pd.read_csv(FINAL_OUTPUT)
else:
    print("Starting new process...")
    df = pd.read_csv(INPUT_CSV)
    if NEW_COLUMN not in df.columns:
        df[NEW_COLUMN] = ""

# 3. print the len output (real-time)
print(f"\n--- Starting Analysis (Total: {len(df)} rows) ---")
print(f"Descriptions will appear below in real-time:")
print("-" * 50)

processed_this_session = 0

for i, row in df.iterrows():
    # Skipping previous columns
    if pd.notna(row[NEW_COLUMN]) and str(row[NEW_COLUMN]).strip() != "" and "Error" not in str(row[NEW_COLUMN]):
        continue
    
    url = row['Poster']
    movie_name = row.get('original_title', 'Unknown Movie')
    
    if not isinstance(url, str) or not url.startswith('http'):
        df.at[i, NEW_COLUMN] = "Invalid URL"
        continue

    try:
        # image process
        response = requests.get(url, stream=True, timeout=15)
        img = Image.open(response.raw).convert("RGB")
        
        with torch.no_grad():
            enc = model.encode_image(img)
            ans = model.answer_question(enc, "Briefly describe this movie poster.", tokenizer)
        
        description = str(ans).strip()
        df.at[i, NEW_COLUMN] = description
        
        # print- real time
        print(f"🎬 Movie: {movie_name}")
        print(f"📝 AI Desc: {description}")
        print("-" * 30)
        
    except Exception as e:
        error_msg = f"Error: {str(e)[:50]}"
        df.at[i, NEW_COLUMN] = error_msg
        print(f"❌ Failed at row {i} ({movie_name}): {error_msg}")

    processed_this_session += 1
    
    # save and clear memory
    if processed_this_session % SAVE_EVERY == 0:
        df.to_csv(FINAL_OUTPUT, index=False, encoding='utf-8-sig')
        clean_vram()
        print(f"\n[Checkpoint] Saved progress up to row {i} to {FINAL_OUTPUT}\n")

# 4. finall save and download link
df.to_csv(FINAL_OUTPUT, index=False, encoding='utf-8-sig')
print("\n" + "="*50)
print("FINISHED! All processed rows are saved.")
print("="*50)

from IPython.display import FileLink
FileLink(FINAL_OUTPUT)

--- Loading Moondream Model... ---


config.json:   0%|          | 0.00/277 [00:00<?, ?B/s]

hf_moondream.py: 0.00B [00:00, ?B/s]

config.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/vikhyatk/moondream2:
- config.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


...

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/99.0 [00:00<?, ?B/s]

Starting new process...

--- Starting Analysis (Total: 4805 rows) ---
Descriptions will appear below in real-time:
--------------------------------------------------
🎬 Movie: Avatar
📝 AI Desc: The movie poster for "Avatar" features a blue-skinned woman's face and a man's face, with the title "AVATAR" prominently displayed.
------------------------------
🎬 Movie: Pirates of the Caribbean: At World's End
📝 AI Desc: A movie poster for "Pirates of the Caribbean: At World's End" features a pirate with a sword and hat amidst smoke, with the title displayed prominently.
------------------------------
🎬 Movie: Spectre
📝 AI Desc: A man in a white tuxedo, holding a gun, stands with arms crossed against a blue background, wearing a red rose on his lapel. The title "SPECTRE" and the number "007" are displayed prominently.
------------------------------
🎬 Movie: The Dark Knight Rises
📝 AI Desc: A dramatic movie poster for "The Dark Knight Rises" features Batman in his iconic black suit, standing ag

/kaggle/working/movies_full_descriptions.csv